# DGM contaminato + 3 archi fisici (architettura dello schema)

- **DGM**: riceve le feature dei nodi e la concatenazione delle 3 topologie (network, context, knowledge) e genera gli archi latenti.
- **Aggregazione**: `HeteroConv` sulle 4 relazioni: i 3 archi fisici con le loro feature (GATv2) e gli archi latenti del DGM (GCN).
- **Classificazione**: MLP sui nodi e classificatore sugli archi.

**Correzioni rispetto a `contaminato1`:**
1. Gli archi latenti ora arrivano davvero alle convoluzioni. Prima l'assegnazione su `data.edge_index_dict` modificava una copia temporanea e la relazione `latent` veniva saltata.
2. Il DGM ora riceve gradiente: le probabilità degli archi scelti vengono passate al `GCNConv` come `edge_weight`. Prima `topk` restituiva solo indici e l'MLP del DGM non veniva mai addestrato.
3. La normalizzazione `log1p` delle feature dei nodi ora viene applicata davvero (stesso problema di `x_dict`).
4. La soglia di Youden viene scelta sul **validation set** e poi applicata al test.


In [ ]:
# Installa PyTorch Geometric
!pip install torch_geometric


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, GCNConv, GATv2Conv
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_fscore_support,
    accuracy_score,
    confusion_matrix,
    roc_curve
)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42) # Cambia questo numero (es. 0, 42, 100) se la prima run parte male

Cella modificata con l'aggiunta del DGM:

In [ ]:
# ==========================================
# 1) Moduli DGM (Prelevati dal Prof)
# ==========================================
class MLP(nn.Module):
    """Feed-forward network per creare lo spazio latente."""
    def __init__(self, dims, dropout=0.0):
        super().__init__()
        layers = []
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                layers.append(nn.ReLU())
                if dropout > 0:
                    layers.append(nn.Dropout(dropout))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class DynamicGraphModule(nn.Module):
    """Generatore di archi latenti dinamici (kNN basato sulle feature)."""
    def __init__(self, input_dim, hidden_dim=32, latent_dim=16, k=5):
        super().__init__()
        self.embed = MLP([2 * input_dim, hidden_dim, latent_dim])
        self.k = int(k)
        self.temperature = nn.Parameter(torch.tensor(1.0))
        
    @staticmethod
    def aggregate_initial_neighbors(x, edge_index):
        if edge_index is None:
            return torch.zeros_like(x)
        src, dst = edge_index
        out = torch.zeros_like(x)
        deg = torch.zeros(x.shape[0], device=x.device, dtype=x.dtype)
        out.index_add_(0, src, x[dst])
        deg.index_add_(0, src, torch.ones_like(src, dtype=x.dtype))
        return out / deg.clamp_min(1.0).unsqueeze(-1)
        
    def forward(self, x, initial_edge_index=None):
        # 1. Contesto iniziale basato sul grafo originale
        context = self.aggregate_initial_neighbors(x, initial_edge_index)
        
        # 2. Embedding nello spazio latente
        z = self.embed(torch.cat([x, context], dim=-1))

        # 3. Calcolo delle distanze e probabilità di collegamento
        diag_mask = torch.eye(x.shape[0], dtype=torch.bool, device=x.device)
        dist = torch.cdist(z, z, p=2).pow(2)
        scale = torch.exp(self.temperature.clamp(-5.0, 5.0))
        logits = -dist * scale
        logits = logits.masked_fill(diag_mask, -1e9)

        probs = torch.softmax(logits, dim=1)
        probs = probs.masked_fill(diag_mask, 0.0)

        # 4. Estrazione dei k-vicini per creare i nuovi archi
        k = min(max(1, self.k), x.shape[0] - 1)
        idx = torch.topk(probs, k=k, dim=1, largest=True).indices
        src = torch.arange(x.shape[0], device=x.device).view(-1, 1).expand_as(idx).reshape(-1)
        dst = idx.reshape(-1)
        edge_index = torch.stack([src, dst], dim=0)

        return z, edge_index, probs

# ==========================================
# 2) HeteroEdgeClassifier (DGM Contaminato + 3 archi fisici)
# ==========================================
class HeteroEdgeClassifier(nn.Module):
    def __init__(self, in_channels_ip, hidden_channels=64, out_channels=32, num_classes=2, dgm_k=5):
        super().__init__()
        
        # 1. DGM: Riceve direttamente le feature base dei nodi
        self.dgm = DynamicGraphModule(
            input_dim=in_channels_ip, 
            hidden_dim=hidden_channels, 
            latent_dim=16, 
            k=dgm_k
        )

        # 2. AGGREGAZIONE: Legge l'output del DGM (latent) e tutti i 3 archi fisici contemporaneamente
        self.conv1 = HeteroConv({
            ('ip', 'network', 'ip'): GATv2Conv(-1, hidden_channels, edge_dim=2, add_self_loops=False),
            ('ip', 'context', 'ip'): GATv2Conv(-1, hidden_channels, edge_dim=2, add_self_loops=False),
            ('ip', 'knowledge', 'ip'): GATv2Conv(-1, hidden_channels, edge_dim=2, add_self_loops=False),
            ('ip', 'latent', 'ip'): GCNConv(-1, hidden_channels), 
        }, aggr='sum')

        self.bn1_ip = nn.BatchNorm1d(hidden_channels)

        self.conv2 = HeteroConv({
            ('ip', 'network', 'ip'): GATv2Conv(hidden_channels, hidden_channels, edge_dim=2, add_self_loops=False),
            ('ip', 'context', 'ip'): GATv2Conv(hidden_channels, hidden_channels, edge_dim=2, add_self_loops=False),
            ('ip', 'knowledge', 'ip'): GATv2Conv(hidden_channels, hidden_channels, edge_dim=2, add_self_loops=False),
            ('ip', 'latent', 'ip'): GCNConv(hidden_channels, hidden_channels), 
        }, aggr='sum')

        self.bn2_ip = nn.BatchNorm1d(hidden_channels)

        # 3. CLASSIFICAZIONE
        self.lin_node = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels * 2),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(hidden_channels * 2, out_channels),
            nn.ReLU()
        )

        self.lin_edge = nn.Sequential(
            nn.Linear(out_channels * 2, out_channels),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(out_channels, out_channels),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(out_channels, num_classes)
        )

    def forward(self, data):
        # 1. Estrazione feature iniziali dei nodi IP
        x_ip = data.x_dict['ip']
        device = x_ip.device
        
        # 2. AGGREGAZIONE 1 (Strategia A.2 - Unione delle 3 topologie)
        
        # Recupero dei 3 set di archi fisici (usando .get() con tensori vuoti di fallback per sicurezza)
        edge_net = data.edge_index_dict.get(('ip', 'network', 'ip'), torch.empty((2, 0), dtype=torch.long, device=device))
        edge_ctx = data.edge_index_dict.get(('ip', 'context', 'ip'), torch.empty((2, 0), dtype=torch.long, device=device))
        edge_knw = data.edge_index_dict.get(('ip', 'knowledge', 'ip'), torch.empty((2, 0), dtype=torch.long, device=device))
        
        # Concatenazione dei tre archi per creare una topologia iniziale unica e completa
        initial_edges_aggregated = torch.cat([edge_net, edge_ctx, edge_knw], dim=1)
        
        # 3. IL DGM GENERA I NUOVI ARCHI DINAMICI:riceve le Node Features e la concatenzione
        z_latent, learned_edge_index, edge_probs = self.dgm(x_ip, initial_edges_aggregated)

        # 4. Nuovi archi come "Edge set 4".
        # ATTENZIONE: data.edge_index_dict viene ricostruito da PyG a ogni accesso,
        # quindi assegnarci dentro non ha effetto. Lavoriamo su una copia esplicita.
        edge_index_dict = dict(data.edge_index_dict)
        edge_index_dict[('ip', 'latent', 'ip')] = learned_edge_index

        # topk restituisce solo indici (non differenziabili): per far arrivare il gradiente
        # al DGM, pesiamo ogni arco latente con la sua probabilità di collegamento.
        lat_src, lat_dst = learned_edge_index
        edge_weight_dict = {('ip', 'latent', 'ip'): edge_probs[lat_src, lat_dst]}

        # Preparazione del dizionario delle feature degli archi da passare alle convoluzioni
        edge_attr_dict = {
            ('ip', 'network', 'ip'): data['ip', 'network', 'ip'].edge_attr,
            ('ip', 'context', 'ip'): data['ip', 'context', 'ip'].edge_attr,
            ('ip', 'knowledge', 'ip'): data['ip', 'knowledge', 'ip'].edge_attr,
        }

        # 5. AGGREGAZIONE 2: Convoluzioni sulle 4 relazioni
        x_dict = self.conv1(data.x_dict, edge_index_dict,
                            edge_attr_dict=edge_attr_dict, edge_weight_dict=edge_weight_dict)
        x_dict['ip'] = self.bn1_ip(x_dict['ip'])
        x_dict['ip'] = F.relu(x_dict['ip'])

        x_dict = self.conv2(x_dict, edge_index_dict,
                            edge_attr_dict=edge_attr_dict, edge_weight_dict=edge_weight_dict)
        x_dict['ip'] = self.bn2_ip(x_dict['ip'])
        x_dict['ip'] = F.relu(x_dict['ip'])

        x_dict['ip'] = self.lin_node(x_dict['ip'])
        
        return x_dict

    def edge_classify(self, z_src, z_dst):
        z_cat = torch.cat([z_src, z_dst], dim=-1)
        return self.lin_edge(z_cat)

# ==========================================
# 3) Focal Loss (Corretta per il grave sbilanciamento)
# ==========================================
def focal_loss(logits, targets, alpha=0.25, gamma=2.0):
    # Calcolo dell'errore base
    ce_loss = F.cross_entropy(logits, targets, reduction='none')
    pt = torch.exp(-ce_loss)

    # Peso dinamico alpha_t:
    # Se è un attacco (1), assegniamo un peso enorme (es. 0.95)
    # Se è traffico benigno (0), riduciamo il peso al 5% (0.05) per non farci sommergere
    alpha_t = torch.where(
        targets == 1, 
        torch.tensor(alpha, device=targets.device), 
        torch.tensor(1.0 - alpha, device=targets.device)
    )

    return (alpha_t * (1 - pt)**gamma * ce_loss).mean()

In [ ]:
# ==========================================
# 4) ADDESTRAMENTO
# ==========================================
def train_edge_classifier(model, graphs, epochs=10, device='cuda', alpha=0.25, gamma=2.0):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for data in graphs:
            data = data.to(device)
            optimizer.zero_grad()

            x_dict = model(data)
            z_ip = x_dict['ip']

            loss_graph = 0.0
            for rel_type in data.edge_types:
                if 'edge_label' not in data[rel_type]: continue
                labels = data[rel_type].edge_label
                if labels.numel() == 0: continue

                src, dst = data[rel_type].edge_index
                logits = model.edge_classify(z_ip[src], z_ip[dst])

                loss_graph += focal_loss(logits, labels, alpha=alpha, gamma=gamma)

            if isinstance(loss_graph, torch.Tensor):
                loss_graph.backward()
                optimizer.step()
                total_loss += loss_graph.item()

        print(f"[Epoch {epoch}/{epochs}] Loss: {total_loss / len(graphs):.4f}")

In [ ]:
%matplotlib inline
################################
# 5) VALUTAZIONE: soglia scelta sul VALIDATION, applicata al TEST
################################
@torch.no_grad()
def compute_scores(model, graphs, device='cuda'):
    """Restituisce (punteggi di attacco, etichette vere) per tutti gli archi dei grafi."""
    model.eval()
    model.to(device)
    all_scores, all_labels = [], []

    for data in graphs:
        data = data.to(device)
        x_dict = model(data)
        z_ip = x_dict['ip']

        for rel_type in data.edge_types:
            if 'edge_label' not in data[rel_type]: continue
            labels = data[rel_type].edge_label
            if labels.numel() == 0: continue

            src, dst = data[rel_type].edge_index
            logits = model.edge_classify(z_ip[src], z_ip[dst])

            probs = F.softmax(logits, dim=-1)[:, 1]
            all_scores.extend(probs.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return np.array(all_scores), np.array(all_labels)


def choose_threshold(model, val_graphs, device='cuda'):
    """Soglia di Youden (max TPR - FPR) calcolata SOLO sul validation set."""
    scores, labels = compute_scores(model, val_graphs, device)
    if len(np.unique(labels)) < 2:
        print("Il validation set contiene una sola classe: uso soglia 0.5")
        return 0.5
    fpr, tpr, thresholds = roc_curve(labels, scores)
    threshold = thresholds[np.argmax(tpr - fpr)]
    print(f"Soglia scelta sul VALIDATION (Youden): {threshold:.6f}  |  AUC validation = {roc_auc_score(labels, scores):.4f}")
    return threshold


def evaluate_edge_classifier(model, test_graphs, threshold, device='cuda'):
    """Valuta sul TEST con una soglia fissata in precedenza (mai calcolata sul test)."""
    scores, labels = compute_scores(model, test_graphs, device)
    if len(np.unique(labels)) < 2:
        print("Only one class in test set => AUC=0.5 by definition.")
        return

    auc = roc_auc_score(labels, scores)
    preds = (scores > threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    accuracy = accuracy_score(labels, preds)
    cm = confusion_matrix(labels, preds)

    print(f"\nTEST (soglia {threshold:.6f}):")
    print(f"AUC={auc:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}, Accuracy={accuracy:.4f}")
    print("Confusion Matrix:\n", cm)


###############################
# 6) Data-Loading Helpers (Normalizzazione Completa)
###############################
def load_single_graph(directory, filename):
    path = os.path.join(directory, filename)
    data = torch.load(path, weights_only=False)

    # 1. NORMALIZZAZIONE NODI (in_degree / out_degree arrivano a migliaia)
    # Si assegna a data['ip'].x: data.x_dict è una copia ricostruita a ogni accesso.
    if 'x' in data['ip']:
        data['ip'].x = torch.log1p(torch.abs(data['ip'].x))

    for rel_type in data.edge_types:
        # 2. SISTEMAZIONE LABEL E CAST A LONG (Intero per la CrossEntropy)
        if 'edge_label' in data[rel_type]:
            lbl = data[rel_type].edge_label
            lbl = torch.where(lbl < 0, torch.tensor(0, device=lbl.device), lbl)
            lbl = torch.where(lbl > 1, torch.tensor(1, device=lbl.device), lbl)
            data[rel_type].edge_label = lbl.long()

        # 3. NORMALIZZAZIONE ARCHI
        if 'edge_attr' in data[rel_type]:
            attr = data[rel_type].edge_attr
            data[rel_type].edge_attr = torch.log1p(torch.abs(attr))

    return data


def load_graphs(directory):
    files = sorted(f for f in os.listdir(directory) if f.endswith('.pt'))
    return [load_single_graph(directory, f) for f in files]


# ==========================================
# 7) MAIN SCRIPT KAGGLE
# ==========================================
if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    # Cartelle generate da train_for_3edge.ipynb (train + val) e test_for_3edge.ipynb (test)
    base_dir  = "/kaggle/input/datasets/irenelontano/grafitesi"
    train_dir = os.path.join(base_dir, "train_hetero_graphs_pt")
    val_dir   = os.path.join(base_dir, "val_hetero_graphs_pt")
    test_dir  = os.path.join(base_dir, "test_hetero_graphs_pt")

    train_graphs = load_graphs(train_dir)
    val_graphs   = load_graphs(val_dir)
    test_graphs  = load_graphs(test_dir)
    print(f"Loaded {len(train_graphs)} train, {len(val_graphs)} validation, {len(test_graphs)} test graphs.")

    # in_channels_ip = 4 (Community size, In-Degree, Out-Degree, Centrality)
    model = HeteroEdgeClassifier(in_channels_ip=4, hidden_channels=64, out_channels=32, dgm_k=2)

    print("\n--- Inizio Addestramento ---")
    train_edge_classifier(model, train_graphs, epochs=70, device=device)

    print("\n--- Scelta della soglia sul Validation Set ---")
    threshold = choose_threshold(model, val_graphs, device=device)

    print("\n--- Valutazione sul Test Set ---")
    evaluate_edge_classifier(model, test_graphs, threshold, device=device)
